# **Categorical Encoding**

---
# **📌 TL;DR: Encoding Cheat Sheet**

| Encoding Method   | Handles High Cardinality | Keeps Order | Avoids Leakage | Interpretable | Suits Tree Models | Suits Linear Models |
| ----------------- | ------------------------ | ----------- | -------------- | ------------- | ----------------- | ------------------- |
| One-Hot           | ❌                        | ❌           | ✅              | ✅             | ✅                 | ✅                   |
| Ordinal           | ❌                        | ✅           | ✅              | ✅             | ✅                 | ❌ (if not ordered)  |
| Target / Mean     | ✅                        | ❌           | ❌              | ❌             | ✅                 | ✅                   |
| Frequency / Count | ✅                        | ❌           | ✅              | ✅             | ✅                 | ✅                   |
| Binary            | ✅                        | ❌           | ✅              | ❌             | ✅                 | ✅                   |
| Leave-One-Out     | ✅                        | ❌           | ✅ (somewhat)   | ❌             | ✅                 | ✅                   |
| Hashing           | ✅✅                       | ❌           | ✅              | ❌             | ✅                 | ✅                   |


| Data Scenario                                | Recommended Encoding                       | ✅ Pros                                                                            | ⚠️ Cons                                                                                          |
| -------------------------------------------- | ------------------------------------------ | --------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------ |
| **Low cardinality, nominal**                 | One-Hot Encoding                           | - Simple and intuitive<br>- No ordinal assumption<br>- Works with most models     | - Increases dimensionality<br>- Can lead to sparsity                                             |
| **Ordered categories**                       | Ordinal Encoding                           | - Preserves order<br>- Low dimensionality<br>- Fast and simple                    | - Can mislead models into interpreting linear distance if not true                               |
| **High cardinality**                         | Target / Frequency / Binary Encoding       | - Reduces dimensionality<br>- Captures info-rich patterns                         | - Risk of overfitting (Target)<br>- Less interpretable (Binary)<br>- Requires careful validation |
| **Streaming / unknown categories**           | Hashing Encoding                           | - Fixed output size<br>- Fast and memory-efficient<br>- No need to store mappings | - Hash collisions<br>- Not interpretable<br>- Cannot recover original values                     |
| **Small dataset + sensitive to overfitting** | One-Hot or Count Encoding                  | - Avoids target leakage<br>- Easy to validate<br>- Interpretable                  | - OHE may cause high dimensionality<br>- Count encoding may lose nuance                          |
| **Tree models**                              | Any encoding, but OHE or Count works great | - Trees are scale-invariant<br>- Handle categorical splits natively with OHE      | - May still benefit from dimensionality reduction on very large categories                       |
| **Linear models**                            | OHE or Target Encoding (with care)         | - Works well after scaling<br>- Captures interactions if encoded correctly        | - Target encoding can leak information<br>- Scaling often required for performance               |

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

In [13]:
# Create a simple example dataset with multiple categorical features
data = {
    'color': ['red', 'blue', 'green', 'blue', 'red'],
    'size': ['S', 'M', 'L', 'XL', 'M'],
    'shape': ['circle', 'square', 'triangle', 'circle', 'square']
}
X = pd.DataFrame(data)

# Sample target variable for demonstration
y = np.array([1, 0, 1, 0, 1])

---
# **🔢 One Hot Encoder**

- Transforms each category into a separate binary column
- E.g., color = red, blue, green → `color_red`, `color_blue`, `color_green`	

| 🔧 When to Use                                        | 🚫 When Not to Use                                           |
| ----------------------------------------------------- | ------------------------------------------------------------ |
| Tree-based models (e.g., RF, XGBoost) handle OHE well | When cardinality (number of categories) is **high** (>15–20) |
| If categories are **nominal** (no order)              | Can lead to high dimensionality and sparsity                 |


In [14]:
from sklearn.preprocessing import OneHotEncoder

# Initialize OneHotEncoder to output dense array
ohe = OneHotEncoder(sparse_output=False)

# Fit and transform all categorical columns
encoded = ohe.fit_transform(X)

# Create DataFrame for encoded features with proper column names
encoded_X = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(X.columns))

print(encoded_X)

   color_blue  color_green  color_red  size_L  size_M  size_S  size_XL  \
0         0.0          0.0        1.0     0.0     0.0     1.0      0.0   
1         1.0          0.0        0.0     0.0     1.0     0.0      0.0   
2         0.0          1.0        0.0     1.0     0.0     0.0      0.0   
3         1.0          0.0        0.0     0.0     0.0     0.0      1.0   
4         0.0          0.0        1.0     0.0     1.0     0.0      0.0   

   shape_circle  shape_square  shape_triangle  
0           1.0           0.0             0.0  
1           0.0           1.0             0.0  
2           0.0           0.0             1.0  
3           1.0           0.0             0.0  
4           0.0           1.0             0.0  


---
# **🏷️ Label Encoder**

- Assigns a unique integer to each category.
- E.g., {'Male': 0, 'Female': 1}

| 🔧 **When to Use**                                                               | 🚫 **When Not to Use**                                                                                           |
| -------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------- |
| When the categorical variable has a **natural order** (ordinal)                  | When the variable is **nominal** and has **no meaningful order**                                                 |
| With **tree-based models**, which are generally insensitive to encoded magnitude | With **linear or distance-based models** (e.g., Logistic Regression, KNN) — model may assume false relationships |
| For **binary categorical variables** like `'Yes'/'No'`                           | When encoding leads to **misinterpretation of relationships** between categories                                 |
| When **dimensionality reduction** is important                                   | When interpretability is critical and categories are better separated via One-Hot                                |


In [ ]:
from sklearn.preprocessing import LabelEncoder

# For label encoding, apply per column because LabelEncoder works on 1D arrays
df_label_encoded = X.copy()

for col in X.columns:
    le = LabelEncoder()
    df_label_encoded[col] = le.fit_transform(X[col])

print(df_label_encoded)

   color  size  shape
0      2     2      0
1      0     1      1
2      1     0      2
3      0     3      0
4      2     1      1


---
# **⬆️ Ordinal Encoding**

- Replaces each category with an integer (e.g., `Low=1`, `Medium=2`, `High=3`)

| 🔧 When to Use                                                 | 🚫 When Not to Use                                                             |
| -------------------------------------------------------------- | ------------------------------------------------------------------------------ |
| If categories have a **natural order** (e.g., education level) | Never use if **no order** exists — models may assume order that isn’t real     |
| Simple models like Linear Regression or tree models            | Misleads distance-based or regularized models (e.g., logistic regression, KNN) |

In [16]:
from sklearn.preprocessing import OrdinalEncoder

# Initialize OrdinalEncoder
oe = OrdinalEncoder()

# Fit and transform all categorical columns
ordinal_encoded = oe.fit_transform(X)

# Convert to DataFrame with original columns
ordinal_df = pd.DataFrame(ordinal_encoded, columns=X.columns)

print(ordinal_df)

   color  size  shape
0    2.0   2.0    0.0
1    0.0   1.0    1.0
2    1.0   0.0    2.0
3    0.0   3.0    0.0
4    2.0   1.0    1.0


---
# **🃏 Binary Encoding**

- Converts categories to binary, then splits binary digits into separate columns
E.g., 5 → 101 → col_1=1, col_2=0, col_3=1

| 🔧 When to Use                                    | 🚫 When Not to Use                            |
| ------------------------------------------------- | --------------------------------------------- |
| Large cardinality (hundreds of unique categories) | If interpretability is important              |
| Better than OHE for high cardinality              | Some ML models may misinterpret binary splits |


In [17]:
import category_encoders as ce

# Initialize BinaryEncoder
be = ce.BinaryEncoder(cols=['color', 'size', 'shape'])

# Fit and transform
binary_encoded = be.fit_transform(X)

print(binary_encoded)

   color_0  color_1  size_0  size_1  size_2  shape_0  shape_1
0        0        1       0       0       1        0        1
1        1        0       0       1       0        1        0
2        1        1       0       1       1        1        1
3        1        0       1       0       0        0        1
4        0        1       0       1       0        1        0


---
# **🎯 Target / Mean Encoding**

- Replaces each category with the mean target value for that category
- E.g., `city = Tokyo` → 0.85 (85% churn rate in Tokyo)

| Scenario                                                          | Why                                               |
| ----------------------------------------------------------------- | ------------------------------------------------- |
| ✅ When you have **high-cardinality categorical variables**          | Avoids exploding dimensions (like in One-Hot)     |
| ✅ With **linear models, GBMs (like LightGBM, XGBoost)**             | Encodes signal from target directly into features |
| ✅ On **large datasets** (thousands of rows per category)            | Reduces risk of overfitting                       |
| ✅ In **Kaggle-style modeling** where small performance gains matter | Adds signal that improves accuracy                |
| ❌ On small datasets                            | Prone to overfitting (encoding leaks target info)       |
| ❌ Without cross-validation or smoothing        | Can leak data from training to target, misleading model |
| ❌ On categorical variables unrelated to target | Adds noise, not signal                                  |



In [ ]:
import category_encoders as ce
import numpy as np

# Sample target variable for demonstration
y = np.array([1, 0, 1, 0, 1])

# Initialize TargetEncoder
te = ce.TargetEncoder(cols=['color', 'size', 'shape'])

# Fit on df and target y, then transform
target_encoded = te.fit_transform(X, y)

print(target_encoded)

      color      size     shape
0  0.656740  0.652043  0.585815
1  0.514889  0.585815  0.585815
2  0.652043  0.652043  0.652043
3  0.514889  0.521935  0.585815
4  0.656740  0.585815  0.585815


---
# **🔄️ Frequency / Count Encoding**

- Replace categories with how often they occur in the dataset
- E.g., `Singapore=122`, `London=44`

| Scenario                                                               | Why                                                                     |
| ---------------------------------------------------------------------- | ----------------------------------------------------------------------- |
| ✅ When you have **high-cardinality categorical variables**             | Avoids One-Hot dimensionality explosion; encodes compactly              |
| ✅ With **tree-based models** like Random Forests or XGBoost            | Trees handle numerical splits well; frequency helps define split points |
| ✅ When you need a **fast, leakage-free encoding**                      | Doesn’t use the target → no risk of target leakage                      |
| ✅ For **baseline models** or rapid prototyping                         | Simple to implement and often "good enough" for initial modeling        |
| ❌ When frequency has **no meaningful relation to the target variable** | May introduce irrelevant or misleading patterns                         |
| ❌ With **linear or distance-based models** without scaling             | Raw frequencies may distort coefficients or distance calculations       |
| ❌ If you need **high interpretability**                                | Frequencies are not always meaningful or intuitive to explain           |


In [1]:
import category_encoders as ce
import pandas as pd

# Example DataFrame
X = pd.DataFrame({
    'color': ['red', 'blue', 'green', 'red', 'green', 'blue', 'blue'],
    'size': ['S', 'M', 'L', 'S', 'L', 'M', 'S']
})

# Initialize FrequencyEncoder
fe = ce.CountEncoder(cols=['color', 'size'])  # CountEncoder is used for frequency

# Fit and transform the data
freq_encoded = fe.fit_transform(X)

print(freq_encoded)

   color  size
0      2     3
1      3     2
2      2     2
3      2     3
4      2     2
5      3     2
6      3     3


---
# **#️⃣ Hashing Encoding**

- Maps categories to fixed-size columns using a hash function
- Useful in real-time/streaming pipelines

| 🔧 When to Use                                          | 🚫 When Not to Use                                       |
| ------------------------------------------------------- | -------------------------------------------------------- |
| Very high-cardinality features (e.g., city names, URLs) | Can't interpret or reverse-transform encoded data        |
| When memory/performance matters                         | Risk of **hash collisions** (diff categories same value) |


In [ ]:
import category_encoders as ce

# Initialize HashingEncoder
he = ce.HashingEncoder(cols=['color', 'size', 'shape'], n_components=5)

# Transform the data (no fit needed since it's stateless)
hash_encoded = he.fit_transform(X)

print(hash_encoded)

   col_0  col_1  col_2  col_3  col_4
0      0      0      2      0      1
1      1      1      0      0      1
2      2      0      0      0      1
3      1      0      0      0      2
4      0      1      1      0      1
